<a href="https://colab.research.google.com/github/Abhilipsha-behera22/Agentic-AI-Pipeline-to-Automate-EDA/blob/main/Agentic_ai_Pipeline_to_automate_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Fix the 'zstd' error by installing the compression utility
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Run the Ollama installation script again
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the Ollama server in the background
import subprocess
import time
import os

# Check if ollama was installed correctly
if os.path.exists("/usr/local/bin/ollama") or os.path.exists("/usr/bin/ollama"):
    print("Ollama installed successfully. Starting server...")
    subprocess.Popen(["ollama", "serve"])
    time.sleep(15) # Give it 15 seconds to fully wake up

    # 4. Pull the Mistral model
    !ollama pull mistral
else:
    print("Ollama installation failed. Please check the error messages above.")

# 5. Install Python dependencies
!pip install langchain-community pandas matplotlib seaborn

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,895 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,935 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packag

In [ ]:
# Replace 'archive.zip' with the actual name of your uploaded zip file
!unzip -q 'archive (17).zip' -d market_data

# List files to confirm everything is there
!ls market_data

etfs  stocks  symbols_valid_meta.csv


In [ ]:
import pandas as pd
from langchain_community.llms import Ollama

# 1. Load a specific stock file from the extracted folder
df = pd.read_csv("market_data/stocks/AAPL.csv")
df['Date'] = pd.to_datetime(df['Date']) # Ensure dates are handled correctly

# 2. Extract Metadata (The Blueprint)
schema_info = {
    "columns": df.columns.tolist(),
    "dtypes": df.dtypes.astype(str).to_dict(),
    "shape": df.shape,
    "missing_values": df.isnull().sum().to_dict()
}

# 3. Initialize LLM
llm = Ollama(model="mistral")

/tmp/ipykernel_11714/1190171321.py:17: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

coder_prompt = f"""
You are a Senior Data Scientist.
Dataset metadata: {schema_info}

Write Python code using pandas and seaborn to:
1. Plot a 'Close' price trend line using the 'Date' column.
2. Create a distribution plot of the 'Volume' column.
3. Generate a correlation heatmap of all numerical columns.

Return ONLY executable Python code. Do not include markdown backticks or explanations.
"""

# Generate code
generated_code = llm.invoke(coder_prompt)

# Clean and Execute
clean_code = generated_code.replace("```python", "").replace("```", "").strip()

print("--- AI IS RUNNING THE FOLLOWING CODE ---")
print(clean_code)

# Run the AI's code
try:
    exec(clean_code)
except Exception as e:
    print(f"Error executing AI code: {e}")

--- AI IS RUNNING THE FOLLOWING CODE ---
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('your_dataset.csv', parse_dates=['Date'])

# Plot 'Close' price trend line using the 'Date' column
plt.figure(figsize=(12,6))
sns.lineplot(x='Date', y='Close', data=df)
plt.title('Close Price Trend Line')
plt.show()

# Create a distribution plot of the 'Volume' column
plt.figure(figsize=(10,6))
sns.distplot(df['Volume'], kde=True)
plt.xlabel('Volume')
plt.ylabel('Frequency')
plt.title('Volume Distribution')
plt.show()

# Generate a correlation heatmap of all numerical columns
corr = df[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].corr()
mask = np.triu(corr)

fig, ax = plt.subplots(figsize=(10,7))
sns.heatmap(mask, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()
Error executing AI code: [Errno 2] No such file or directory: 'your_dataset.csv'


In [ ]:
# Create a statistical summary string
stats_summary = df.describe().to_string()

analyst_prompt = f"""
You are a Senior Financial Analyst. Analyze these statistics for this stock:
{stats_summary}

Provide:
- A summary of the stock's volatility.
- Observations on trading volume trends.
- A final 'Senior Recommendation' for a potential investor.
"""

insights = llm.invoke(analyst_prompt)

print("\n--- SENIOR ANALYST REPORT ---")
print(insights)


--- SENIOR ANALYST REPORT ---
 Summary of the Stock's Volatility:
The stock appears to exhibit significant volatility based on the provided data. The high and low prices show a large range, with the maximum being 327.85 (high) and the minimum at 0.1964 (low). The standard deviation of closing prices is 58.41, indicating substantial fluctuations in the stock's price over time.

Observations on Trading Volume Trends:
The trading volume varies greatly, with a daily average of approximately 8.58 million shares traded. However, the data does not provide information on volume trends over longer periods, making it difficult to determine if there is an upward or downward trend in trading activity.

Final Senior Recommendation for a Potential Investor:
Given the high volatility of this stock and the lack of clear trends in trading volume, potential investors should approach with caution. It's crucial to conduct additional research on factors influencing the stock's performance, such as market 

In [ ]:
import os

def run_accuracy_test(llm, dataframe, generated_code, insights):
    print("🚀 Starting Agentic Pipeline Validation...\n")
    score = 0
    total_tests = 4

    # TEST 1: Code Executability
    print("Test 1: Code Integrity...")
    try:
        # We try to compile the code to see if it's valid Python
        compile(generated_code, '<string>', 'exec')
        print("✅ Pass: Code is syntactically correct.")
        score += 1
    except Exception as e:
        print(f"❌ Fail: Code has syntax errors. {e}")

    # TEST 2: Grounding (Anti-Hallucination)
    print("\nTest 2: Numerical Grounding...")
    # Check if the AI's report mentions numbers that actually exist in the data
    actual_max = round(dataframe['Close'].max(), 2)
    if str(int(actual_max)) in insights:
        print(f"✅ Pass: Analyst recognized the correct Max Price ({actual_max}).")
        score += 1
    else:
        print(f"⚠️ Warning: Analyst might be hallucinating or using generalities.")

    # TEST 3: Library Usage
    print("\nTest 3: Technical Requirement Check...")
    requirements = ['plt', 'sns', 'pd']
    found_reqs = [req for req in requirements if req in generated_code]
    if len(found_reqs) >= 2:
        print(f"✅ Pass: Coder Agent used correct libraries ({found_reqs}).")
        score += 1
    else:
        print("❌ Fail: Coder Agent missed essential libraries.")

    # TEST 4: Context Awareness
    print("\nTest 4: Schema Alignment...")
    # Check if the code mentions actual columns from our dataset
    col_matches = [col for col in dataframe.columns if col in generated_code]
    if len(col_matches) > 0:
        print(f"✅ Pass: Code is specifically tailored to these columns: {col_matches}")
        score += 1
    else:
        print("❌ Fail: Code is generic and not mapped to dataset columns.")

    # FINAL GRADE
    final_percent = (score / total_tests) * 100
    print(f"\n--- FINAL ACCURACY SCORE: {final_percent}% ---")
    if final_percent == 100:
        print("Excellent! Your Agent is production-ready.")
    else:
        print("Good start! Refine your prompts to hit 100%.")

# To run the test, just pass in your variables from the previous steps:
run_accuracy_test(llm, df, clean_code, insights)

🚀 Starting Agentic Pipeline Validation...

Test 1: Code Integrity...
✅ Pass: Code is syntactically correct.

Test 2: Numerical Grounding...
✅ Pass: Analyst recognized the correct Max Price (327.2).

Test 3: Technical Requirement Check...
✅ Pass: Coder Agent used correct libraries (['plt', 'sns', 'pd']).

Test 4: Schema Alignment...
✅ Pass: Code is specifically tailored to these columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

--- FINAL ACCURACY SCORE: 100.0% ---
Excellent! Your Agent is production-ready.
